In [0]:
#Load cleansed silver table from unity catalog
silver_df = spark.read.table("workspace.default.silver_yellow_taxi")

print(f"Silver Base Records Available: {silver_df.count():,}")
display(silver_df.select("pickup_date", "pickup_hour", "pickup_borough", "fare_amount", "trip_distance").limit(5))

Silver Base Records Available: 2,495,712


pickup_date,pickup_hour,pickup_borough,fare_amount,trip_distance
2026-01-01,0,Manhattan,10.0,1.17
2026-01-01,0,Manhattan,9.3,1.39
2026-01-01,0,Manhattan,32.4,2.96
2026-01-01,0,Manhattan,19.1,2.29
2026-01-01,0,Manhattan,11.4,1.73


### **Business Questions Answered by Gold Tables**

#### **Table 1: Hourly Demand & Revenue Patterns**
* **Q1:** Which hours of the day generate the highest trip demand and volume across New York City?
* **Q2:** How does total revenue correlate with hour of day, and when does fare yield peak?
* **Q3:** Do passenger trip lengths (distance in miles and duration in minutes) fluctuate between daytime rush hours and late-night hours?

#### **Table 2: Borough-to-Borough Route Dynamics**
* **Q4:** Which pickup-to-dropoff borough routes handle the heaviest taxi traffic?
* **Q5:** Which specific corridors generate the highest average fares and longest trip distances?
* **Q6:** What proportion of taxi trips stay within the same borough versus crossing into other boroughs?

#### **Table 3: Payment Economics & Tipping Behavior**
* **Q7:** What percentage of riders pay with credit cards versus cash?
* **Q8:** How does average tip percentage vary by pickup borough for card transactions?

## **GOLD TABLE **1****

In [0]:
from pyspark.sql.functions import count,sum,round,avg,col

In [0]:
Hourly_demand_df = silver_df.groupby("pickup_hour").agg(
    count("*").alias("total_trips"),
    round(sum("total_amount"),2).alias("total_revenue"),
    round(avg("fare_amount"),2).alias("avg_fare"),
    round(avg("trip_distance"),2).alias("avg_distance_miles"),
    round(avg("trip_duration_minutes"),2).alias("avg_duration_minutes")
).orderBy("pickup_hour")
display(Hourly_demand_df)

pickup_hour,total_trips,total_revenue,avg_fare,avg_distance_miles,avg_duration_minutes
0,58033,1778800.7,20.53,4.09,13.55
1,37443,1017727.32,17.88,3.4,12.17
2,25484,653832.44,16.69,3.08,11.5
3,17311,466450.3,17.9,3.45,11.85
4,12174,392851.01,22.97,4.95,16.14
5,15954,604682.38,28.89,6.84,22.19
6,32472,1071422.46,25.1,5.66,21.97
7,64672,1847579.8,20.69,4.14,19.2
8,90744,2451988.33,19.08,3.46,17.74
9,108238,2916083.3,18.83,3.29,17.07


In [0]:
Hourly_demand_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_hourly_demand")

print("Saved: workspace.default.gold_hourly_demand")

Saved: workspace.default.gold_hourly_demand


In [0]:
# Q1: Peak hours by trip count
display(
    Hourly_demand_df.select("pickup_hour", "total_trips")
    .orderBy(col("total_trips").desc())
)

pickup_hour,total_trips
18,181267
17,180457
16,171408
15,168633
14,157521
19,155409
13,147026
12,141749
20,141207
21,137828


**Q1: Which hours generate the highest demand?**
  * **Peak Demand:** **5:00 PM – 6:59 PM (Hours 17 & 18)** with **180,457** and **181,267** trips respectively.
  * **Lowest Demand:** **4:00 AM (Hour 4)** with just **12,174 trips**.

In [0]:
# To answer Q2 (Top Revenue Hours):# Q2: Top revenue hours
display(
    Hourly_demand_df.select("pickup_hour", "total_revenue")
    .orderBy(col("total_revenue").desc())
)

pickup_hour,total_revenue
17,5353419.54
16,5348228.91
18,5169179.77
15,4941974.48
14,4600810.85
19,4548280.16
20,4150051.42
13,4084510.46
21,3995279.07
12,3874477.16


**Q2: When does revenue peak and how does it correlate with volume?**
  * **Peak Revenue:** Peaks at **5:00 PM (Hour 17)** at **$5,353,419.54**, directly aligning with volume.
  * Total hourly revenue rises steadily from 6:00 AM onward until 6:00 PM.


In [0]:
# Q3: Trip distance, duration, and fare across hours
display(
    Hourly_demand_df.select(
        "pickup_hour", 
        "avg_distance_miles", 
        "avg_duration_minutes", 
        "avg_fare"
    ).orderBy("pickup_hour")
)

pickup_hour,avg_distance_miles,avg_duration_minutes,avg_fare
0,4.09,13.55,20.53
1,3.4,12.17,17.88
2,3.08,11.5,16.69
3,3.45,11.85,17.9
4,4.95,16.14,22.97
5,6.84,22.19,28.89
6,5.66,21.97,25.1
7,4.14,19.2,20.69
8,3.46,17.74,19.08
9,3.29,17.07,18.83


**Q3: Do trip distance and duration fluctuate between daytime rush hours and late-night?**
  * **Yes, significantly:** 
    * Daytime rush hour (Hour 17–18) trips are shorter in distance (~**2.96 to 3.11 miles**) due to dense intra-borough traffic.
    * Early morning (Hour 5:00 AM) trips have the longest average distance (**6.84 miles**) and duration (**22.19 minutes**), driven by airport and inter-borough runs, yielding the highest average fare of **$28.89**.

### **GOLD TABLE 2**

In [0]:

# Business Rule: Physical Borough Corridor Filtering
# Raw zone references map unassigned/outside coordinates to 'Unknown' or 'N/A'.
# While retained in Silver for financial reconciliation, they are excluded from Gold
# corridor reporting to ensure traffic and distance metrics reflect authentic NYC boroughs.



invalid_boroughs = ["N/A", "Unknown"]

gold_borough_metrics_df = silver_df.filter(
    col("pickup_borough").isNotNull() & 
    col("dropoff_borough").isNotNull() &
    (~col("pickup_borough").isin(invalid_boroughs)) &
    (~col("dropoff_borough").isin(invalid_boroughs))
).groupBy("pickup_borough", "dropoff_borough").agg(
    count("*").alias("total_trips"),
    round(sum("total_amount"), 2).alias("total_revenue"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance_miles")
).orderBy(col("total_trips").desc())

# Save to Unity Catalog
gold_borough_metrics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_borough_metrics")

print("Saved clean: workspace.default.gold_borough_metrics")
display(gold_borough_metrics_df)

Saved clean: workspace.default.gold_borough_metrics


pickup_borough,dropoff_borough,total_trips,total_revenue,avg_fare,avg_distance_miles
Manhattan,Manhattan,2041355,4.370505919E7,13.23,1.78
Queens,Manhattan,137676,1.136475148E7,57.23,13.91
Manhattan,Queens,55002,3632977.24,47.23,11.09
Queens,Queens,53681,2311253.04,34.3,7.36
Manhattan,Brooklyn,52215,2336145.45,33.47,7.01
Queens,Brooklyn,45321,3028493.08,54.61,13.24
Brooklyn,Brooklyn,22986,552473.29,22.61,3.89
Manhattan,Bronx,14711,579195.12,35.46,8.6
Brooklyn,Manhattan,13222,549862.76,38.03,9.46
Bronx,Manhattan,11348,395143.06,33.89,8.6


In [0]:
# Q4: Top 5 corridors by trip volume
display(
    gold_borough_metrics_df.select(
        "pickup_borough", "dropoff_borough", "total_trips", "total_revenue"
    ).orderBy(col("total_trips").desc()).limit(5)
)

pickup_borough,dropoff_borough,total_trips,total_revenue
Manhattan,Manhattan,2041355,4.370505919E7
Queens,Manhattan,137676,1.136475148E7
Manhattan,Queens,55002,3632977.24
Queens,Queens,53681,2311253.04
Manhattan,Brooklyn,52215,2336145.45


* **Q4: Which routes handle the heaviest taxi traffic?**
  * **Top Route:** **Manhattan to Manhattan** dominates overall volume with **2,041,355 trips** generating over **$43.7M**.
  * **Second Top Route:** **Queens to Manhattan** (airport inbound traffic) with **137,676 trips** generating **$11.36M+**.

In [0]:
# Q5: Highest average fare corridors (minimum 100 trips for statistical relevance)
display(
    gold_borough_metrics_df.filter(col("total_trips") >= 100)
    .select("pickup_borough", "dropoff_borough", "avg_fare", "avg_distance_miles", "total_trips")
    .orderBy(col("avg_fare").desc())
    .limit(10)
)

pickup_borough,dropoff_borough,avg_fare,avg_distance_miles,total_trips
Queens,EWR,155.27,33.98,200
Queens,Staten Island,102.42,26.52,312
Manhattan,EWR,95.32,17.64,4654
Manhattan,Staten Island,75.3,18.59,200
Queens,Bronx,59.56,15.74,6701
Queens,Manhattan,57.23,13.91,137676
Queens,Brooklyn,54.61,13.24,45321
Brooklyn,Bronx,54.59,17.04,2809
Bronx,Brooklyn,54.16,16.97,2947
Manhattan,Queens,47.23,11.09,55002


**Q5: Which corridors generate the highest average fares and longest distances?**
  * **Queens to EWR:** **$155.27 avg fare** (33.98 avg miles).
  * **Queens to Staten Island:** **$102.42 avg fare** (26.52 avg miles).
  * **Manhattan to EWR:** **$95.32 avg fare** (17.64 avg miles).
  * **Manhattan to Staten Island:** **$75.30 avg fare** (18.59 avg miles).
  * **Queens to Bronx:** **$59.56 avg fare** (15.74 avg miles).

In [0]:
from pyspark.sql.functions import when

# Q6: Intra-Borough vs. Inter-Borough Mobility Ratio
# Measures what percentage of taxi trips remain inside the pickup borough
# versus crossing boundaries into another borough.

mobility_df = silver_df.filter(
    col("pickup_borough").isNotNull() & 
    col("dropoff_borough").isNotNull() &
    (~col("pickup_borough").isin(["N/A", "Unknown"])) &
    (~col("dropoff_borough").isin(["N/A", "Unknown"]))
).withColumn(
    "trip_type",
    when(col("pickup_borough") == col("dropoff_borough"), "Intra-Borough")
    .otherwise("Inter-Borough")
)

total_valid_trips = mobility_df.count()

display(
    mobility_df.groupBy("trip_type").agg(
        count("*").alias("total_trips"),
        round((count("*") / total_valid_trips) * 100, 2).alias("percentage_share"),
        round(sum("total_amount"), 2).alias("total_revenue")
    ).orderBy(col("total_trips").desc())
)

trip_type,total_trips,percentage_share,total_revenue
Intra-Borough,2119401,85.58,4.660386208E7
Inter-Borough,357250,14.42,2.389061313E7


* **Q6: What proportion of taxi trips stay within the same borough versus crossing into other boroughs?**
  * **Intra-Borough (Within same borough):** **85.58%** of all trips (**2,119,401 trips**), generating **$46.60M** in revenue.
  * **Inter-Borough (Crossing borough lines):** **14.42%** of trips (**357,250 trips**), generating **$23.89M** in revenue.
 

### GOLD TABLE 3

In [0]:

# Gold Table 3: Payment Economics & Tipping Behavior
# Maps payment IDs to human-readable names and computes transaction shares.

from pyspark.sql.functions import when

silver_payment_df = silver_df.withColumn(
    "payment_method",
    when(col("payment_type") == 1, "Credit Card")
    .when(col("payment_type") == 2, "Cash")
    .when(col("payment_type") == 3, "No Charge")
    .when(col("payment_type") == 4, "Dispute")
    .otherwise("Other")
)

gold_payment_tipping_df = silver_payment_df.groupBy("payment_method").agg(
    count("*").alias("total_transactions"),
    round(sum("total_amount"), 2).alias("total_revenue"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(sum("tip_amount"), 2).alias("total_tips_recorded"),
    round(avg("tip_amount"), 2).alias("avg_tip_recorded"),
    round(avg("tip_percentage"), 2).alias("avg_tip_pct")
).orderBy(col("total_transactions").desc())

# Save to Unity Catalog
gold_payment_tipping_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_payment_tipping")

print("Saved clean: workspace.default.gold_payment_tipping")
display(gold_payment_tipping_df)

Saved clean: workspace.default.gold_payment_tipping


payment_method,total_transactions,total_revenue,avg_fare,total_tips_recorded,avg_tip_recorded,avg_tip_pct
Credit Card,2173349,6.385677966E7,19.34,8917946.22,4.1,25.01
Cash,287690,7249948.54,19.53,120.29,0.0,0.0
Dispute,26405,796866.38,23.91,44.09,0.0,0.0
No Charge,8268,212842.35,19.89,1.69,0.0,0.0


In [0]:
 
# Q7: Payment Method Adoption & Cashless Mobility Share

total_txns = silver_payment_df.count()

display(
    silver_payment_df.groupBy("payment_method").agg(
        count("*").alias("total_transactions"),
        round((count("*") / total_txns) * 100, 2).alias("transaction_share_pct"),
        round(sum("total_amount"), 2).alias("total_revenue")
    ).orderBy(col("total_transactions").desc())
)

payment_method,total_transactions,transaction_share_pct,total_revenue
Credit Card,2173349,87.08,6.385677966E7
Cash,287690,11.53,7249948.54
Dispute,26405,1.06,796866.38
No Charge,8268,0.33,212842.35


**Q7: What percentage of riders pay with credit cards versus cash?**
* **Credit Card:** **87.08%** (2,173,349 transactions)
* **Cash:** **11.53%** (287,690 transactions)
* **Dispute:** **1.06%** (26,405 transactions)
* **No Charge:** **0.33%** (8,268 transactions)

In [0]:
# Q8: Average tip percentage by pickup borough for card transactions
display(
    silver_df.filter(
        (col("payment_type") == 1) & 
        col("pickup_borough").isNotNull() & 
        (~col("pickup_borough").isin(["N/A", "Unknown"]))
    ).groupBy("pickup_borough").agg(
        round(avg("tip_percentage"), 2).alias("avg_tip_pct"),
        count("*").alias("total_card_trips")
    ).orderBy(col("avg_tip_pct").desc())
)

pickup_borough,avg_tip_pct,total_card_trips
Manhattan,26.29,1907144
Queens,19.76,200385
EWR,12.89,20
Staten Island,4.24,47
Brooklyn,3.43,44517
Bronx,0.63,17600



* **Q8: How does average tip percentage vary by pickup borough for card transactions?**
  * **Manhattan:** **26.29%** (1,907,144 trips)
  * **Queens:** **19.76%** (200,385 trips)
  * **EWR:** **12.89%** (20 trips)
  * **Staten Island:** **4.24%** (47 trips)
  * **Brooklyn:** **3.43%** (44,517 trips)
  * **Bronx:** **0.63%** (17,600 trips)